In [3]:
import pandas as pd
import zipfile
import os
import shutil # Import shutil for removing directories

# Path to the zip file
zip_file_path = 'Pokemon Data Analysis Tutorial.zip'

# Directory to extract the files to
# Assuming the zip contains a top-level folder with the same name as the zip (without extension)
extracted_folder_name = os.path.splitext(zip_file_path)[0] # "Pokemon Data Analysis Tutorial"
extract_path = extracted_folder_name # Extract into this folder

# Create the extraction directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Files extracted to {extract_path}")

# Construct the full paths to the CSV files, accounting for a nested folder inside the zip
pokemon_csv_path = os.path.join(extract_path, extracted_folder_name, 'pokemon.csv')
combats_csv_path = os.path.join(extract_path, extracted_folder_name, 'combats.csv')

print(f"Attempting to load pokemon_df from: {pokemon_csv_path}")
print(f"Attempting to load combats_df from: {combats_csv_path}")

# Load the datasets
pokemon_df = pd.read_csv(pokemon_csv_path)
combats_df = pd.read_csv(combats_csv_path)

print("Datasets loaded successfully.")

# Clean up extracted files as per Comment #4
try:
    shutil.rmtree(extract_path)
    print(f"Cleaned up extracted directory: {extract_path}")
except OSError as e:
    print(f"Error: {e.filename} - {e.strerror}.")

Files extracted to Pokemon Data Analysis Tutorial
Attempting to load pokemon_df from: Pokemon Data Analysis Tutorial/Pokemon Data Analysis Tutorial/pokemon.csv
Attempting to load combats_df from: Pokemon Data Analysis Tutorial/Pokemon Data Analysis Tutorial/combats.csv
Datasets loaded successfully.
Cleaned up extracted directory: Pokemon Data Analysis Tutorial


### Pokemon DataFrame Head

In [4]:
display(pokemon_df.head())

,#,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,4,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,5,Charmander,Fire,NaN,39,52,43,60,50,65,1,False


### Combats DataFrame Head

In [5]:
display(combats_df.head())

,First_pokemon,Second_pokemon,Winner
0,266,298,298
1,702,701,701
2,191,668,668
3,237,683,683
4,151,231,151


### Checking for missing values and data types in `pokemon_df`

In [ ]:
print(pokemon_df.info())
display(pokemon_df.head())

### Handling Missing Values

In [ ]:
# Fill missing Name for Pokemon #62 (Primeape)
pokemon_df.loc[pokemon_df['#'] == 62, 'Name'] = pokemon_df.loc[pokemon_df['#'] == 62, 'Name'].fillna('Primeape')

# Handle NaN values in 'Type 2' by replacing them with 'None'
pokemon_df['Type 2'] = pokemon_df['Type 2'].fillna('None')

print("Missing values handled successfully.")
print("\nUpdated info for pokemon_df after handling missing values:")
print(pokemon_df.info())

### Calculate Win Percentage

In [6]:
# Calculate wins for each Pokémon
# This creates a Series where index is Pokemon ID and value is number of wins
win_counts = combats_df['Winner'].value_counts()

# Calculate total battles for each Pokémon
# Concatenate 'First_pokemon' and 'Second_pokemon' columns to get all participants
all_participants = pd.concat([combats_df['First_pokemon'], combats_df['Second_pokemon']])
battle_counts = all_participants.value_counts()

# Create a DataFrame for win percentages
# Start with all unique Pokémon IDs from the main pokemon_df to ensure all are included
all_pokemon_ids = pokemon_df['#'].unique()
win_percentage_df = pd.DataFrame(all_pokemon_ids, columns=['#'])

# Merge win counts
win_percentage_df = win_percentage_df.merge(
    win_counts.rename('Wins'), left_on='#', right_index=True, how='left'
)
# Merge battle counts
win_percentage_df = win_percentage_df.merge(
    battle_counts.rename('Battles'), left_on='#', right_index=True, how='left'
)

# Fill NaN values (for Pokémon that didn't win or battle) with 0
win_percentage_df['Wins'] = win_percentage_df['Wins'].fillna(0).astype(int)
win_percentage_df['Battles'] = win_percentage_df['Battles'].fillna(0).astype(int)

# Calculate win percentage
# Handle division by zero for Pokémon with 0 battles by filling with 0
win_percentage_df['Win_Percentage'] = (
    win_percentage_df['Wins'] / win_percentage_df['Battles'] * 100
).fillna(0)

# Select only the necessary columns for merging back into the main DataFrame
win_percentage_df = win_percentage_df[['#', 'Win_Percentage']]

# Merge win_percentage_df with pokemon_df
pokemon_df = pd.merge(pokemon_df, win_percentage_df, on='#', how='left')

print("Win percentages calculated and merged successfully (optimized).")
display(pokemon_df.head())

Win percentages calculated and merged successfully (optimized).


,#,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,Win_Percentage
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False,27.819549
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False,38.016529
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False,67.424242
3,4,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False,56.000000
4,5,Charmander,Fire,NaN,39,52,43,60,50,65,1,False,49.107143
